In [ ]:
words = open('../names.txt', 'r').read().splitlines()

In [ ]:
words[:10]

In [ ]:
len(words)

In [ ]:
min(len(w) for w in words)

In [ ]:
max(len(w) for w in words)

In a bigram language model, we're only ever looking at 2 characters at a time. 

So we're looking at 1 character we are given, and trying to predicted the next character in a sequence.

We're ignoring the fact that we may have a lot more information. It's a very simple and weak language model.

In [ ]:
b = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

In [ ]:
sorted(b.items(), key = lambda kv: -kv[1])

In [ ]:
import torch

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)    # Bigram counts
                                                # 26 alphabet characters + 1 special character

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [ ]:
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='black')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='black')
plt.axis('off')

In [ ]:
N[0]

Now convert raw counts to probabilities.

In [ ]:
p = N[0].float() # get all characters in first row (count of words that start with each char)
p = p / p.sum() # probability of any character to be the first character of a word; sums to 1
p

Now we can sample from this distribution

In [ ]:
# workaround to get the same results as the video, which uses an older pytorch
def sample(p, g):
    # replicates old torch.multinomial(p, 1) so results match the video
    u = torch.rand(1, generator=g, dtype=torch.float64)
    return int(torch.searchsorted(p.double().cumsum(0), u))

In [ ]:
g = torch.Generator().manual_seed(2147483647)
ix = sample(p,g)
itos[ix]

In [ ]:
g = torch.Generator().manual_seed(2147483647) # Generator makes everything deterministic so I get the same results as Karpathy's tutorial
p = torch.rand(3, generator=g)
p = p / p.sum()
p

In [ ]:
torch.multinomial(p, num_samples=20, replacement=True, generator=g) # torch.multinomial draws samples: 
                                                                    # takes the tensor of probability distributions (p)

In [ ]:
# Let's make the below block more efficient by making the P probability distribution matrix
# (instead of calculating it every loop)
P = N.float()
P /= P.sum(1, keepdim=True) # normalize by column vec (sum all rows and normalize) (normalize rows)

# P is really the parameters of our bigram language model. It summarizes the statistics of our bigrams.

In [ ]:
P = (N+1).float() # MODEL SMOOTHING: add a count of 1 to every table cell. Stops loss function from being infinity on "impossible" words.
P /= P.sum(1, keepdim=True) 

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    
    out = []
    ix = 0
    while True:
        
        p = P[ix]
        #p = N[ix].float() 
        #p = p / p.sum() # convert count to probability distribution
        #p = torch.ones(27) / 27.0 # sample from uniform distribution to compare (completely untrained model)
        
        ix = sample(p,g) # sample distribution to get next char (row index)
        out.append(itos[ix]) # append sampled next char to out list
        if ix == 0: # if end token: end word
            break
    print(''.join(out))

Next, let's try to evaluate the quality of the model.

We want to summarize the quality of the model into a single number: how good is it at predicting, let's say the training set.

So now in the training set we can evaluate the training loss, which tells us about the quality of this model in a single number (just like in micrograd).

According to standard Maximum Likelihood Estimation literature, we can get the quality of the model by taking the *likelihood*: the product of all these probabilities. If the quality of the model is high, the product of these probabilities (the likelihood) should be high.

The likelihood tells us the probability of the dataset, assigned by the model we've trained. That is a measure of quality.

For convenience, because the probabilities we're working with are low floats between 0-1, people generally work with not the likelihood but the log likelihood. 0 log likelihood = optimal.

But the thing is, we'd like a loss function. A loss function has the semantics that low is good (we want to minimize loss). So we want to invert the log likelihood. That's what gives us the *negative log likelihood* -- a very nice loss function. One more thing people like to do is to to normalize the nll: make it an average rather than a sum.

In [ ]:
log_likelihood = 0.0
n = 0

for w in words:
#for w in ["theo"]: # you can evaluate any single word
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2] # if uniform probability (untrained), we'd assume each prob is 1/27 or ~.37%
                           # if you have a very good model, probs should be around 1.00, beause you'd assume the model is very good at predicting (especially on training set)
        logprob = torch.log(prob)
        log_likelihood += logprob # recall from hs math: log(a*b*c) = log(a) + log(b) + log(c)
        n += 1
        #print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')
        
print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}') # average nll -- this is our loss functixon we will use

Now we have a loss function (average nll) that we want to minimize. That will improve the model quality.

We do this minimization by tuning the model parameters -- the probabilities, defined in our table.

What's coming up: we'll not keep the parameters in our table explicitly, but rather they will be calculated by a neural network.

So now we'll cast the problem of bigram character-level language modeling into the neural network framework. Different approach, but we'll end up in a similar spot.

The neural network (still a bigram character-level LM) will do this:

input: a single character

$\rightarrow$ NN with some parameters W $\rightarrow$

output: the probability distribution over the next character in a sequence.

And: we're going to be able to evaluate any setting of the parameters of the NN with our loss function. 

So: we're going to look at its probability distributions, and use the labels (the next char in the bigram, ch2). That way we can see the model's assigned probability to the actual ch2 (from the training set), and tweak the parameters so the probability is high (the loss is low). So we're going to use gradient-based optimization to tune the parameters of the network by minimizing the loss function.

In [ ]:
# create the training set of all the bigrams (x, y)
xs, ys = [], [] # inputs, targets (labels)

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        print(ch1, ch2)
        xs.append(ix1)
        ys.append(ix2)
        
xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
xs

In [ ]:
ys

In [ ]:
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes=27).float()
xenc

In [ ]:
xenc.shape

In [ ]:
plt.imshow(xenc)

In [ ]:
W = torch.randn((27, 27)) # W is neuron
xenc @ W # wx + b, but b = 0

In [ ]:
(xenc @ W)[3, 13] # gives the firing rate of the 13th neuron on the 3rd input

Now we have 27 inputs, fed into 27 neurons. A single linear layer. Each neuron does wx.